## Variable sources

The example source is pulsed and has a sweeping PA. Both the pulse and sweep are at a frequency of 10 Hz. The simulated light curve is sin(phase * pi)**2 and the simulated PA sweep is fixed-rate counterclockwise motion, with PD = 50%.

In [1]:
import numpy as np
import leakagelib

>>> PyXSPEC is not installed, you will no be able to use it.


We'll load the data as usual, and cut to within 280 arcsec of the center

In [2]:
datas = leakagelib.IXPEData.load_all_detectors_with_path("data", "pulse")
for data in datas:
    data.iterative_centroid_center()
    data.retain(data.evt_energies > 2)
    data.retain(data.evt_energies < 8)

>>> Reading (in memory) /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/xrt/bcf/vign/ixpe_d1_obssim20240101_vign_v013.fits...


However, we plan to implement phase weights. LeakageLib does not automatically record phases. You should calculate them yourself (e.g. with PINT), then replace the `evt_times` field with these phases. For this mock data, the frequency is exactly 10 Hz, so the phases are just $10 t$.

In [3]:
for data in datas:
    # It's convenient to overwrite the "times" list with a list of phases. This source sweeps with frequency of 10 Hz, so multiplying by 10 gives the phase
    data.evt_times *= 10
    data.evt_times = np.fmod(data.evt_times, 1)

Now we create the point source and background sources as usual, setting their spectra. For this fit, we'll fix the background polarization

In [4]:
settings = leakagelib.FitSettings(datas)
settings.apply_circular_roi(280)

settings.add_point_source("src")
settings.fix_flux("src", 1)

settings.add_background("bkg")
settings.fix_qu("bkg", (0, 0))
settings.set_initial_flux("bkg", 1)

settings.set_spectrum("bkg", lambda e: e**-2.5)
settings.set_spectrum("src", lambda e: e**-1.5)

6345 events were cut for being outside the region of interest.
>>> Reading (in memory) /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/gpd/cpf/arf/ixpe_d1_obssim20240101_v013.arf...
>>> Using cached xVignetting object at /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/xrt/bcf/vign/ixpe_d1_obssim20240101_vign_v013.fits...
>>> Using cached xEffectiveArea object at /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/gpd/cpf/arf/ixpe_d1_obssim20240101_v013.arf...
>>> Using cached xVignetting object at /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/xrt/bcf/vign/ixpe_d1_obssim20240101_vign_v013.fits...


To implement phase weights, we need to tell `LeakageLib` the source's light curve. Note that normalization doesn't matter.

.. warning::
    If you cut based on phase e.g. for an on-off fit, you will need to supply the `duty_cycle` argument in `set_lightcurve`.

In [5]:
settings.set_lightcurve("src", lambda ph: np.sin(ph * np.pi)**2)

Now we can perform the fit and see what we get.

In [6]:
fitter = leakagelib.Fitter(settings)
print(fitter)
const_result = fitter.fit()
const_result

Data set pulse DU 1 had no exposure map loaded. Please load an exposure map if you are fitting to events in the vignetted portion.


FITTED PARAMETERS:
Source	Param
src:	q
src:	u
bkg:	f

FIXED PARAMETERS:
Source	Param	Value
src:	f	1
bkg:	q	0
bkg:	u	0

0.0336759090423584
0.033966064453125
0.03292417526245117
0.03297281265258789
0.03510093688964844
0.03344321250915527
0.035125017166137695
0.03382992744445801
0.03224778175354004
0.03306102752685547
0.03398585319519043
0.03350710868835449
0.03446507453918457
0.03324699401855469
0.0328221321105957
0.03603482246398926
0.03548002243041992
0.03470492362976074
0.034921884536743164
0.033442020416259766
0.03330588340759277
0.032425880432128906
0.031961917877197266
0.03346991539001465
0.03476905822753906
0.034131765365600586
0.03370499610900879
0.03326892852783203
0.033476829528808594
0.033004045486450195
0.0342409610748291
0.035073041915893555
0.034004926681518555
0.03309917449951172
0.03349494934082031
0.03477978706359863
0.03503012657165527
0.0352630615234375
0.035536766052246094
0.0336000919342041
0.03314518928527832
0.0336151123046875
0.035529136657714844
0.035404205322265

KeyboardInterrupt: 

The best-fit polarization degree was quite low. This is because the source's true PA sweeps, and we modeled it as constant. We need to make a sweeping PA model.

This model will have two parameters: a constant PD and a PA at phase zero. Let's create those parameters. The `add_param` function allows us to do this, and give initial fit values and bounds.

In [7]:
settings.add_param("sweep-PD", 0.1, [0, 1])
settings.add_param("sweep-PA", 0, [-np.pi, np.pi])

Now we need to create a function which takes in the phase and gives the expected Q and U polarization, and tell the fitter to use this function.

In [8]:
def model_fn(ph, fit_data, param_array):
    pd = fit_data.param_to_value(param_array, "sweep-PD")
    pa = fit_data.param_to_value(param_array, "sweep-PA")
    q = pd * np.cos(2 * (ph * 2*np.pi + pa))
    u = pd * np.sin(2 * (ph * 2*np.pi + pa))
    return q, u
settings.set_model_fn("src", model_fn)

Note that `set_model_fn` automatically tells the fitter to no longer use the src q and src u params. When we re-do the fit, it will use the sweep-PD and sweep-PA params instead.

In [9]:
fitter = leakagelib.Fitter(settings)
print(fitter)
sweep_result = fitter.fit()
sweep_result

Data set pulse DU 1 had no exposure map loaded. Please load an exposure map if you are fitting to events in the vignetted portion.


FITTED PARAMETERS:
Source	Param
bkg:	f
None:	sweep-PD
None:	sweep-PA

FIXED PARAMETERS:
Source	Param	Value
src:	q	0
src:	u	0
src:	f	1
bkg:	q	0
bkg:	u	0



FitResult:
	f (bkg) = 2.1937 +/- 0.0421
	sweep-PD = 0.4666 +/- 0.0708
	sweep-PA = 0.0232 +/- 0.0782

Polarization:
Likelihood 18970.099283443626, dof 15756
Optimization terminated successfully.

The PD of the sweeping fit was higher. The likelihood was also higher, indicating the sweep model provided a better fit. The true source polarization had PD = 0.5 and PA = 0, which the fit agrees with.